# Notebook 6: DistilBERT variants for better generalisation (trained on SNLI only, evaluated on SNLI and MNLI)
**Kaggle: GPU T4, Internet ON, attach Datasets: `nli-data`, `nli-teacher` (Notebook 4 output), `nli-lr-results` (Notebook 1 output:
needs `hyponly_train_oof.csv` and `preds_lr_hyponly_snli_val.csv` / `_snli_test.csv`).**

Arms (all use the baseline's exact recipe: lr 5e-5, 3 epochs, batch 32, warmup 10%, wd 0.01, fp16, seeds 42/43/44):
- `kd`         : distillation from the fine-tuned Llama teacher's soft labels (+ gold CE), SNLI 100K only.
- `poe`        : product of experts with the hypothesis-only LR (out-of-fold log-probs); the bias expert is dropped at test time.
- `ls`         : control: gold labels with label smoothing 0.1 and no teacher (is the kd gain Llama's knowledge or just soft targets?).
- `kd_generic` : like `kd`, plus Llama-labelled pairs built from generic text (Notebook 5). Add it to `RUN_ARMS` once that Dataset is attached.

Hyper-parameters are chosen on **SNLI validation only** with a pre-declared score = mean(val accuracy, val-hard accuracy), where
"hard" = validation rows the hypothesis-only LR gets wrong. MNLI is never used for training, tuning or selection.
Runtime: about 9 minutes per run (kd: 5 runs, poe: 4 runs).

In [1]:
import glob, json, os, re, sys, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup, set_seed

RUN_ARMS = ["kd_generic"]             # earlier sessions ran kd, poe, ls, kd_generic. kd_extra needs the Dataset from Notebook 4b (snli_extra*.csv)
LS_HP = {"eps": 0.1}
META_HYP = r"(?i)\b(?:the|this|that|given) (?:text|passage|article|sentence|statement|review|description|headline|excerpt|paragraph|story)\b|^(?:it (?:appears|seems) that|based on|according to the)"
KD_GENERIC_HP = {"T": 2.0, "alpha": 0.5}   # used by kd_generic if the kd grid was not run in this session
N_GENERIC = None                    # None = use all generic pairs
RUN_SEEDS = [42]                    # e.g. [42] to run a single seed
MODEL_ONLY = True                 # True: retrain + save the seed-42 model only (no predictions); use with RUN_SEEDS=[42]

def find(pattern):
    h = glob.glob(os.environ.get("NLI_ROOT", "/kaggle/input") + "/**/" + pattern, recursive=True)
    assert h, f"not found: {pattern} (is the right Dataset attached?)"
    return h[0]

ROOT = Path(find("nli_common.py")).parent; sys.path.insert(0, str(ROOT)); DATA = str(ROOT / "data")
from nli_common import LABELS, SEED, load_split, save_run
OUT = Path(os.environ.get("NLI_OUT", "/kaggle/working/results/distilbert_variants")); OUT.mkdir(parents=True, exist_ok=True)
CKPT = Path("/kaggle/working/ckpt"); CKPT.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "turn on the GPU"
dev = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

MODEL = "distilbert-base-uncased"
LR, EPOCHS, BATCH, WARMUP, WD, CLIP = 5e-5, 3, 32, 0.1, 0.01, 1.0
SEEDS, MAX_LEN_TRAIN = [42, 43, 44], 128
MAX_LEN_EVAL = {"snli_val": 128, "snli_test": 128, "mnli_m": 256, "mnli_mm": 256}
GRIDS = {"kd": [{"T": 1.0, "alpha": 0.5}, {"T": 2.0, "alpha": 0.5}, {"T": 2.0, "alpha": 0.2}],
         "poe": [{"w": 0.5}, {"w": 1.0}]}

train = load_split("train_100k", DATA)
EVALS = {n: load_split(n, DATA) for n in ["snli_val", "snli_test", "mnli_m", "mnli_mm"]}
val, test = EVALS["snli_val"], EVALS["snli_test"]

# teacher logits and bias-expert log-probs, aligned to the training rows
teach = pd.read_csv(find("teacher_logits_train_100k.csv"))
assert (teach["id"].to_numpy() == train["id"].to_numpy()).all() and (teach["label"].to_numpy() == train["label"].to_numpy()).all(), "teacher file not aligned with train"
oof = pd.read_csv(find("hyponly_train_oof.csv"))
assert (oof["id"].to_numpy() == train["id"].to_numpy()).all(), "OOF file not aligned with train"
T_LOG = teach[["z_ent", "z_neu", "z_con"]].to_numpy(np.float32)
B_LOGP = np.log(np.clip(oof[["p_ent", "p_neu", "p_con"]].to_numpy(np.float64), 1e-6, 1.0)).astype(np.float32)

# hard subsets = rows the hypothesis-only LR gets wrong
def hard_mask(split_name, df):
    h = pd.read_csv(find(f"preds_lr_hyponly_{split_name}.csv"))
    assert (h["id"].to_numpy() == df["id"].to_numpy()).all()
    return (h["pred"].to_numpy() != h["label"].to_numpy())
HARD = {"snli_val": hard_mask("snli_val", val), "snli_test": hard_mask("snli_test", test)}
print({"train": len(train), "val_hard": int(HARD["snli_val"].sum()), "test_hard": int(HARD["snli_test"].sum())})

GPU: Tesla T4
{'train': 100000, 'val_hard': 3493, 'test_hard': 3503}


## Data, loss and training loop

In [2]:
tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(df, max_len):
    enc = tok(df["premise"].tolist(), df["hypothesis"].tolist(), truncation=True, max_length=max_len)
    return enc["input_ids"], enc["attention_mask"]

class DS(Dataset):
    def __init__(s, ids, am, gold, tlog, blog): s.d = (ids, am, gold, tlog, blog)
    def __len__(s): return len(s.d[0])
    def __getitem__(s, i): return tuple(x[i] for x in s.d)

def collate(b):
    feats = dict(tok.pad([{"input_ids": x[0], "attention_mask": x[1]} for x in b], return_tensors="pt"))
    return (feats, torch.tensor([int(x[2]) for x in b]), torch.tensor(np.stack([x[3] for x in b])), torch.tensor(np.stack([x[4] for x in b])))

def build_train(arm):
    ids, am = tokenize(train, MAX_LEN_TRAIN)
    gold, tlog, blog = train["label"].to_numpy(), T_LOG, B_LOGP
    if arm in ("kd_generic", "kd_extra"):
        g = pd.read_csv(find("generic_pairs*.csv" if arm == "kd_generic" else "snli_extra*.csv"))
        if arm == "kd_generic":       # drop hypotheses that talk about "the text" (an artifact of the generation prompt)
            n0 = len(g); g = g[~g["hypothesis"].str.contains(META_HYP, regex=True)].reset_index(drop=True)
            print(f"generic pairs: {n0:,} -> {len(g):,} after removing meta-text hypotheses")
        if N_GENERIC: g = g.iloc[:N_GENERIC]
        gi, ga = tokenize(g, MAX_LEN_TRAIN)
        ids, am = ids + gi, am + ga
        gold = np.concatenate([gold, -np.ones(len(g), dtype=gold.dtype)])        # -1 = no gold label (distillation only)
        tlog = np.concatenate([tlog, g[["z_ent", "z_neu", "z_con"]].to_numpy(np.float32)])
        blog = np.concatenate([blog, np.zeros((len(g), 3), np.float32)])
        print(f"{arm}: {len(train):,} SNLI + {len(g):,} extra teacher-labelled pairs (no gold labels)")
    return DS(ids, am, gold, tlog, blog)

def loss_fn(logits, gold, tlog, blog, arm, hp):
    logits = logits.float()
    if arm == "ls":                                     # control arm: label smoothing, no teacher
        return F.cross_entropy(logits, gold, label_smoothing=hp["eps"])
    if arm == "poe":                                    # product of experts: main log-probs + weighted bias log-probs
        return F.cross_entropy(F.log_softmax(logits, -1) + hp["w"] * blog, gold)
    T, a = hp["T"], hp["alpha"]                         # kd / kd_generic
    has = (gold >= 0).float()
    ce = F.cross_entropy(logits, gold, ignore_index=-1, reduction="none")                      # 0 where there is no gold label
    kl = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(tlog / T, -1), reduction="none").sum(-1) * T * T
    return ((a * has) * ce + (1 - a * has) * kl).mean()

@torch.no_grad()
def predict(model, df, max_len, bs=128):
    model.eval()
    ids, am = tokenize(df, max_len)
    order = np.argsort([len(x) for x in ids]); out = np.zeros((len(df), 3), np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        feats = {k: v.to(dev) for k, v in tok.pad([{"input_ids": ids[j], "attention_mask": am[j]} for j in idx], return_tensors="pt").items()}
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(**feats).logits
        out[idx] = torch.softmax(lg.float(), -1).cpu().numpy()
    return out

def new_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=3, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)}).to(dev)

def val_scores(model):
    pred = predict(model, val, 128).argmax(1); ok = pred == val["label"].to_numpy()
    return float(ok.mean()), float(ok[HARD["snli_val"]].mean())

def train_run(arm, hp, seed, ds):
    set_seed(seed); model = new_model()
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=True, generator=g, collate_fn=collate, num_workers=2)
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    groups = [{"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)], "weight_decay": WD},
              {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=LR)
    total = EPOCHS * len(loader); sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
    scaler = torch.cuda.amp.GradScaler(); torch.cuda.reset_peak_memory_stats(); hist, train_time = [], 0.0
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); ls = 0.0
        for feats, gold, tlog, blog in loader:
            feats = {k: v.to(dev, non_blocking=True) for k, v in feats.items()}
            gold, tlog, blog = gold.to(dev), tlog.to(dev), blog.to(dev)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**feats).logits
            loss = loss_fn(logits, gold, tlog, blog, arm, hp)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP); scaler.step(opt); scaler.update(); sched.step(); ls += loss.item()
        et = time.time() - t0; train_time += et; va, vh = val_scores(model)
        hist.append({"epoch": ep + 1, "train_loss": ls / len(loader), "val_acc": va, "val_hard_acc": vh, "epoch_s": round(et)})
        print(f"  [{arm} {hp} s{seed}] epoch {ep + 1}/{EPOCHS} loss={ls / len(loader):.4f} val={va:.4f} val-hard={vh:.4f} ({et:.0f}s)")
    return model, {"history": hist, "train_time_s": train_time, "peak_gpu_mem_gb": torch.cuda.max_memory_allocated() / 1e9}

def evaluate_and_save(model, arm, hp, seed, info):
    name = f"distilbert_{arm}_s{seed}"
    config = {"model": MODEL, "arm": arm, "arm_hp": hp, "seed": seed, "train_file": "snli_train_100k" + {"kd_generic": " + generic pairs", "kd_extra": " + extra SNLI pairs"}.get(arm, ""),
              "lr": LR, "epochs": EPOCHS, "batch_size": BATCH, "warmup_frac": WARMUP, "weight_decay": WD, "precision": "fp16 autocast",
              "teacher": "Llama-3.2-1B-Instruct LoRA (Notebook 3)" if arm in ("kd", "kd_generic", "kd_extra") else None,
              "bias_model": "hypothesis-only LR, 5-fold out-of-fold" if arm == "poe" else None,
              "selection_rule": "mean(val acc, val-hard acc) on SNLI validation", "history": info["history"]}
    for es, df in EVALS.items():
        torch.cuda.synchronize(); t0 = time.time(); probs = predict(model, df, MAX_LEN_EVAL[es]); torch.cuda.synchronize(); dt = time.time() - t0
        save_run(OUT, name, es, df, probs, config, {"train_time_s": info["train_time_s"], "peak_gpu_mem_gb": info["peak_gpu_mem_gb"],
                                                     "n_params": sum(p.numel() for p in model.parameters()), "inference_s": dt})

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## Run the arms: small grid on seed 42 (validation only), then 3 seeds with the chosen setting

In [3]:
BEST = {}
for arm in RUN_ARMS:
    print(f"\n===== arm: {arm} =====")
    ds = build_train(arm)
    if arm in GRIDS:
        rows, best = [], None
        for hp in GRIDS[arm]:
            model, info = train_run(arm, hp, 42, ds)
            h = info["history"][-1]; score = 0.5 * (h["val_acc"] + h["val_hard_acc"])
            rows.append({"arm": arm, **hp, "val_acc": h["val_acc"], "val_hard_acc": h["val_hard_acc"], "score": score})
            if best is None or score > best["score"]:
                best = {"hp": hp, "score": score, "info": info}; torch.save(model.state_dict(), CKPT / f"best_{arm}.pt")
            del model; torch.cuda.empty_cache()
        gdf = pd.DataFrame(rows); gdf.to_csv(OUT / f"{arm}_grid.csv", index=False); print(gdf.to_string(index=False))
        BEST[arm] = best["hp"]; print("selected:", best["hp"])
    else:                                     # kd_generic: no grid, reuse the kd setting
        BEST[arm] = LS_HP if arm == "ls" else {"T": 2.0, "alpha": 0.2} if arm in ("kd_generic", "kd_extra") and "kd" not in BEST else BEST.get("kd", KD_GENERIC_HP)
        best = None; print("using:", BEST[arm])
    for seed in (RUN_SEEDS or SEEDS):
        if best is not None and seed == 42:
            model = new_model(); model.load_state_dict(torch.load(CKPT / f"best_{arm}.pt")); info = best["info"]
            model.save_pretrained(OUT / f"distilbert_{arm}_s42_model"); tok.save_pretrained(OUT / f"distilbert_{arm}_s42_model")
        else:
            model, info = train_run(arm, BEST[arm], seed, ds)
            if seed == 42:
                model.save_pretrained(OUT / f"distilbert_{arm}_s42_model"); tok.save_pretrained(OUT / f"distilbert_{arm}_s42_model")
        if not MODEL_ONLY:
            evaluate_and_save(model, arm, BEST[arm], seed, info)
        del model; torch.cuda.empty_cache()
    if (CKPT / f"best_{arm}.pt").exists(): os.remove(CKPT / f"best_{arm}.pt")


===== arm: kd_generic =====
generic pairs: 53,993 -> 47,874 after removing meta-text hypotheses
kd_generic: 100,000 SNLI + 47,874 extra teacher-labelled pairs (no gold labels)
using: {'T': 2.0, 'alpha': 0.2}


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_22/424050314.py:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(); torch.cuda.reset_

  [kd_generic {'T': 2.0, 'alpha': 0.2} s42] epoch 1/3 loss=0.6051 val=0.8461 val-hard=0.7080 (390s)
  [kd_generic {'T': 2.0, 'alpha': 0.2} s42] epoch 2/3 loss=0.2668 val=0.8628 val-hard=0.7266 (392s)
  [kd_generic {'T': 2.0, 'alpha': 0.2} s42] epoch 3/3 loss=0.1579 val=0.8687 val-hard=0.7432 (392s)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Summary (mean ± std over seeds; hard/easy = SNLI-test rows the hypothesis-only LR gets wrong / right)
Baseline reference from Notebook 2 (3 seeds): SNLI test 85.82, MNLI-m 62.79, MNLI-mm 65.05.

In [4]:
def _summary():
    rows = []
    for arm in RUN_ARMS:
        per = {k: [] for k in ["snli_test", "mnli_m", "mnli_mm", "hard", "easy"]}
        for seed in (RUN_SEEDS or SEEDS):
            for es in ["snli_test", "mnli_m", "mnli_mm"]:
                per[es].append(json.load(open(OUT / f"results_distilbert_{arm}_s{seed}_{es}.json"))["metrics"]["accuracy"])
            p = pd.read_csv(OUT / f"preds_distilbert_{arm}_s{seed}_snli_test.csv"); ok = (p["pred"] == p["label"]).to_numpy()
            per["hard"].append(ok[HARD["snli_test"]].mean()); per["easy"].append(ok[~HARD["snli_test"]].mean())
        rows.append({"arm": arm, "hp": str(BEST[arm]), **{k: f"{100 * np.mean(v):.2f} ± {100 * np.std(v, ddof=1):.2f}" for k, v in per.items()}})
    summ = pd.DataFrame(rows); summ.to_csv(OUT / "variants_summary.csv", index=False); print(summ.to_string(index=False))
    print("\nFiles:", sorted(os.listdir(OUT))[:12], "...")
    print("Next: Save Version -> Save & Run All; make a Dataset from the output; the analysis notebook compares against Notebook 2 with McNemar tests.")

if not MODEL_ONLY:
    _summary()
else:
    print('model-only run: no summary; model saved in', OUT)

model-only run: no summary; model saved in /kaggle/working/results/distilbert_variants
